# RIS recall probe — what does the Rechtssatz-only ingest miss?

**Read-only diagnostic.** This notebook does not modify `../data/` or `master_import.ipynb`;
it writes only `../reports/ris_recall_probe.json` and `../reports/ris_recall_probe.md`.

The RIS ingest in `master_import.ipynb` searches `Suchworte` against the API default document
type — **Rechtssätze** (distilled headnotes) — and reaches actual court decisions only by
following each matched Rechtssatz's linked `Entscheidungstexte`. Two things can therefore be
lost: decisions with **no Rechtssatz at all**, and decisions whose **headnote does not contain
the phrase** even though the reasoning does.

This probe measures that gap by running the same six keywords over the same years as a
**full-text decision search**, and diffing the result against the decisions already in
`data/ris_parental_alienation.json`.

Data: **RIS, Bundeskanzleramt Österreich (CC BY 4.0)** — <https://data.bka.gv.at>.
Politeness follows the OGD terms of use: the importer's identifying `User-Agent`, a 1.5 s
delay between every request, 429/5xx backoff, and a bounded six-keyword probe (≈40 requests) —
not a Massendownload.

## 1. Configuration

In [1]:
# RIS recall probe — READ-ONLY diagnostic.
# Does not modify data/ or src/master_import.ipynb; writes only to reports/.
# Data source: RIS, Bundeskanzleramt Österreich (CC BY 4.0) — https://data.bka.gv.at
import json, re, time, collections
from datetime import datetime
from pathlib import Path

import requests

DATA_DIR   = Path("../data")
REPORT_DIR = Path("../reports")
CORPUS     = DATA_DIR / "ris_parental_alienation.json"
MASTER_NB  = Path("master_import.ipynb")

RIS_API = "https://data.bka.gv.at/ris/api/v2.6/Judikatur"
RIS_APP = "Justiz"

# mirrored from master_import.ipynb config so the probe searches exactly the same terms
KEYWORDS_RIS = ["Entfremdung", "elterliche Entfremdung", "Eltern-Kind-Entfremdung",
                "Kindeswohlgefährdung", "Loyalitätskonflikt", "Kontaktverweigerung"]
YEAR_FROM, YEAR_TO = 2000, 2025

REQUEST_DELAY = 1.5     # same politeness budget as the importer — RIS OGD terms of use
PAGE_SIZE     = "OneHundred"
PAGE_SIZE_N   = 100
MAX_PAGES     = 60      # hard ceiling per query; this is a bounded probe, not a Massendownload

# same identifying User-Agent as master_import.ipynb
session = requests.Session()
session.headers.update({
    "User-Agent": "MasterThesis-Research/1.0 (Academic; msmirnov98@gmail.com)",
    "Accept": "application/json",
})

REQUESTS_MADE = 0
print(f"probe: {len(KEYWORDS_RIS)} keywords, {YEAR_FROM}-{YEAR_TO}, delay={REQUEST_DELAY}s")
print(f"corpus under test (read-only): {CORPUS}")

probe: 6 keywords, 2000-2025, delay=1.5s
corpus under test (read-only): ../data/ris_parental_alienation.json


## 2. Primitives

`ris_get` / `ris_parse` keep the same contract as the versions in `master_import.ipynb`
(same session, same delay, same backoff) and add the two documented paging parameters.

In [2]:
# ---- API primitives (same retry/backoff contract as master_import.ris_get) ----

def ris_get(params, retries=3):
    """GET the Judikatur endpoint with 429/5xx backoff. Counts every request."""
    global REQUESTS_MADE
    for attempt in range(retries):
        try:
            time.sleep(REQUEST_DELAY)
            REQUESTS_MADE += 1
            r = session.get(RIS_API, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.HTTPError:
            code = r.status_code
            if code == 429:
                time.sleep(10 * (attempt + 1))
            elif code >= 500:
                return None          # 5xx here means "page past the end of the result set"
            else:
                return None
        except (requests.exceptions.RequestException, ValueError):
            time.sleep(3 * (attempt + 1))
    return None


def ris_parse(data):
    """Copied verbatim from master_import.ipynb."""
    if not data:
        return 0, []
    try:
        res = data["OgdSearchResult"]["OgdDocumentResults"]
        total = int(res.get("Hits", {}).get("#text", 0))
        docs = res.get("OgdDocumentReference", [])
        if isinstance(docs, dict):
            docs = [docs]
        return total, docs
    except (KeyError, TypeError):
        return 0, []


def _item(d, key):
    """A '.item' field that may be str, {'item': ...}, or a list (as in master_import)."""
    val = d.get(key, {})
    if isinstance(val, str):
        return val
    if isinstance(val, dict):
        item = val.get("item", "")
        if isinstance(item, list):
            return "; ".join(str(i) for i in item)
        return str(item) if item else ""
    if isinstance(val, list):
        return "; ".join(str(i) for i in val)
    return ""


def doc_fields(doc):
    """Minimal identity/metadata slice of one OgdDocumentReference."""
    data = doc.get("Data", {})
    meta = data.get("Metadaten", {})
    jud  = meta.get("Judikatur", {})
    just = jud.get("Justiz", {})
    return {
        "id":                 meta.get("Technisch", {}).get("ID", ""),
        "dokumenttyp":        jud.get("Dokumenttyp", ""),
        "geschaeftszahl":     _item(jud, "Geschaeftszahl"),
        "entscheidungsdatum": jud.get("Entscheidungsdatum", ""),
        "gericht":            just.get("Gericht", ""),
        "rechtsgebiete":      _item(just, "Rechtsgebiete"),
        "dokument_url":       meta.get("Allgemein", {}).get("DokumentUrl", ""),
    }


# ---- identity + civil/criminal helpers -------------------------------------

_WS = re.compile(r"\s+")

def norm_gz(gz):
    """Normalised Geschäftszahl identity key: first number, no spaces, lowercase."""
    first = str(gz or "").split(";")[0]
    first = re.sub(r"\(.*?\)", "", first)
    return _WS.sub("", first).lower()


_OS_RE = re.compile(r"\d{1,3}\s*Os\s*\d")   # \b fails on API-format "13Os7/06p"

def is_criminal(rec):
    """Same rule the corpus audit uses: Strafrecht Rechtsgebiet, or an 'Os' senate marker."""
    if "strafrecht" in str(rec.get("rechtsgebiete", "")).lower():
        return True
    return bool(_OS_RE.search(str(rec.get("geschaeftszahl", "")).split(";")[0]))


def year_of(rec):
    try:
        return int(str(rec.get("entscheidungsdatum", ""))[:4])
    except (ValueError, TypeError):
        return None

print("primitives ready")

primitives ready


## 3. Confirmation 1 — what the production importer actually sends

Read straight out of `master_import.ipynb` rather than from memory.

In [3]:
# ---- CONFIRM 1: what does the production importer actually send? -----------
nb = json.loads(MASTER_NB.read_text(encoding="utf-8"))
ris_src = ""
for c in nb["cells"]:
    s = "".join(c["source"])
    if c["cell_type"] == "code" and "def import_ris" in s:
        ris_src = s
        break
import_body = ris_src[ris_src.index("def import_ris"):]
# strip comments — "Dokumenttyp" appears in prose there and would give a false positive
code_only = "\n".join(re.sub(r"#.*$", "", ln) for ln in import_body.splitlines())

print("ris_get(...) calls inside import_ris:")
for line in import_body.splitlines():
    if "ris_get(" in line or "Entscheidungsdatum" in line:
        print("   ", line.strip())

sent = sorted(set(re.findall(r'"([A-Za-z]+)":\s', code_only[:code_only.index("for doc in candidates")])))
print("\nparameter names actually sent by import_ris:", sent)
for p in ["Dokumenttyp", "DokumenteProSeite", "Seitennummer", "Seite"]:
    as_param = bool(re.search('"' + p + '"' + r"\s*:", code_only))
    print(f"   {p:20s} sent as a query parameter: {as_param}")
print("\n=> no Dokumenttyp param -> API default applies -> only Rechtssätze are searched")
print("=> no DokumenteProSeite / Seitennummer -> every query returns the default page of 20")

ris_get(...) calls inside import_ris:
    data = ris_get({"Applikation": RIS_APP, "Suchworte": kw})
    d = ris_get({"Applikation": RIS_APP, "Suchworte": kw,
    "EntscheidungsdatumVon": f"{yr}-01-01",
    "EntscheidungsdatumBis": f"{yr}-12-31"})

parameter names actually sent by import_ris: ['Applikation', 'EntscheidungsdatumBis', 'EntscheidungsdatumVon', 'Suchworte']
   Dokumenttyp          sent as a query parameter: False
   DokumenteProSeite    sent as a query parameter: False
   Seitennummer         sent as a query parameter: False
   Seite                sent as a query parameter: False

=> no Dokumenttyp param -> API default applies -> only Rechtssätze are searched
=> no DokumenteProSeite / Seitennummer -> every query returns the default page of 20


### 3b. Does paging work?

The RIS section's markdown note says the API "always returns 20 results/page and ignores the
page parameter", which is why the importer windows year by year. Test the assumed parameter
name against the documented v2.6 ones.

In [4]:
# ---- CONFIRM 1b: live paging test ------------------------------------------
# The notebook's markdown note blames "broken Seite pagination". Test that name
# against the documented v2.6 names, and prove multi-page retrieval on a query
# that genuinely has more than one page.
probe_kw = "Entfremdung"
base = {"Applikation": RIS_APP, "Suchworte": probe_kw}

t0, d0 = ris_parse(ris_get(base))
print(f"no paging params        : Hits={t0:5d}  returned={len(d0):3d}   <- the 20 ceiling")
t_s, d_s = ris_parse(ris_get({**base, "Seite": 2}))
print(f"Seite=2 (assumed name)  : Hits={t_s:5d}  returned={len(d_s):3d}   <- silently ignored")
t_p, d_p = ris_parse(ris_get({**base, "DokumenteProSeite": PAGE_SIZE, "Seitennummer": 1}))
print(f"DokumenteProSeite=OneHundred&Seitennummer=1: Hits={t_p:5d}  returned={len(d_p):3d}"
      f"   <- whole result set in one call")

# multi-page proof on a large query (the full-text variant of the same keyword)
big = {**base, "Dokumenttyp.SucheInEntscheidungstexten": "true"}
seen, pages = set(), []
for pg in (1, 2, 3):
    tp, dp = ris_parse(ris_get({**big, "DokumenteProSeite": PAGE_SIZE, "Seitennummer": pg}))
    ids = [doc_fields(x)["id"] for x in dp]
    new = len(set(ids) - seen)
    seen |= set(ids)
    pages.append((pg, tp, len(ids), new, len(seen)))
    print(f"  large query, Seitennummer={pg}: Hits={tp:5d}  returned={len(ids):3d}"
          f"  new={new:3d}  cumulative-distinct={len(seen):3d}")

PAGING_WORKS = len(d_p) == t0 and all(p[3] == p[2] for p in pages) and pages[1][2] == PAGE_SIZE_N
print(f"\npaging works with the documented parameter names: {PAGING_WORKS}")
print("(a page past the end of a result set answers HTTP 500 — that is the end-of-results")
print(" signal, not a paging failure)")

no paging params        : Hits=   36  returned= 20   <- the 20 ceiling
Seite=2 (assumed name)  : Hits=   36  returned= 20   <- silently ignored
DokumenteProSeite=OneHundred&Seitennummer=1: Hits=   36  returned= 36   <- whole result set in one call
  large query, Seitennummer=1: Hits=  807  returned=100  new=100  cumulative-distinct=100
  large query, Seitennummer=2: Hits=  807  returned=100  new=100  cumulative-distinct=200
  large query, Seitennummer=3: Hits=  807  returned=100  new=100  cumulative-distinct=300

paging works with the documented parameter names: True
(a page past the end of a result set answers HTTP 500 — that is the end-of-results
 signal, not a paging failure)


## 4. Confirmation 2 — the `Dokumenttyp` flag encoding

The v2.6 documentation lists `Dokumenttyp` with `SucheInRechtssaetzen` / `SucheInEntscheidungstexten`.
Several spellings are plausible; only one actually switches the search, and the others fail
*silently* by falling back to the Rechtssatz default.

In [5]:
# ---- CONFIRM 2: exact query-string encoding of the Dokumenttyp flag --------
def typecount(docs):
    return dict(collections.Counter(doc_fields(x)["dokumenttyp"] for x in docs))

variants = [
    ("(default — no flag)",                          {}),
    ("Dokumenttyp=SucheInEntscheidungstexten",       {"Dokumenttyp": "SucheInEntscheidungstexten"}),
    ("SucheInEntscheidungstexten=true",              {"SucheInEntscheidungstexten": "true"}),
    ("Dokumenttyp.SucheInEntscheidungstexten=true",  {"Dokumenttyp.SucheInEntscheidungstexten": "true"}),
    ("Dokumenttyp.SucheInRechtssaetzen=true",        {"Dokumenttyp.SucheInRechtssaetzen": "true"}),
]
print(f"{'encoding':48s} {'Hits':>7s}  doctypes in first page")
encoding_results = []
for name, extra in variants:
    t, d = ris_parse(ris_get({"Applikation": RIS_APP, "Suchworte": probe_kw,
                              "DokumenteProSeite": "Ten", "Seitennummer": 1, **extra}))
    tc = typecount(d)
    encoding_results.append({"encoding": name, "hits": t, "doctypes": tc})
    print(f"{name:48s} {t:7d}  {tc}")

TE_PARAM = {"Dokumenttyp.SucheInEntscheidungstexten": "true"}
print("\n=> working encoding is the NESTED form: Dokumenttyp.SucheInEntscheidungstexten=true")
print("=> it returns Dokumenttyp 'Text' (Entscheidungstexte), not 'Rechtssatz'")
print("=> the flat forms are silently ignored and fall back to the Rechtssatz default")

encoding                                            Hits  doctypes in first page
(default — no flag)                                   36  {'Rechtssatz': 10}
Dokumenttyp=SucheInEntscheidungstexten                36  {'Rechtssatz': 10}
SucheInEntscheidungstexten=true                       36  {'Rechtssatz': 10}
Dokumenttyp.SucheInEntscheidungstexten=true          807  {'Text': 10}
Dokumenttyp.SucheInRechtssaetzen=true                 36  {'Rechtssatz': 10}

=> working encoding is the NESTED form: Dokumenttyp.SucheInEntscheidungstexten=true
=> it returns Dokumenttyp 'Text' (Entscheidungstexte), not 'Rechtssatz'
=> the flat forms are silently ignored and fall back to the Rechtssatz default


## 5. Paginated search

In [6]:
# ---- paginated search ------------------------------------------------------

def search_all(params, label=""):
    """Page through a Judikatur query until every reported Hit is retrieved.

    Returns (hits_reported, [doc_fields...], truncated_flag).
    Falls back to nothing clever: if paging stops early the flag says so, and the
    caller can decide to year-window instead.
    """
    out, seen = [], set()
    hits = None
    for pg in range(1, MAX_PAGES + 1):
        t, docs = ris_parse(ris_get({**params, "DokumenteProSeite": PAGE_SIZE,
                                     "Seitennummer": pg}))
        if hits is None:
            hits = t
        if not docs:
            break
        for d in docs:
            f = doc_fields(d)
            if f["id"] and f["id"] not in seen:
                seen.add(f["id"])
                out.append(f)
        if len(docs) < PAGE_SIZE_N or len(seen) >= (hits or 0):
            break
    hits = hits or 0
    truncated = len(seen) < hits
    if label:
        print(f"    {label}: Hits={hits:5d}  retrieved={len(seen):5d}"
              f"{'  *** TRUNCATED ***' if truncated else ''}")
    return hits, out, truncated


def year_window(params):
    """Fallback: same query split year by year (used only if paging caps out)."""
    out, seen, total = [], set(), 0
    for yr in range(YEAR_FROM, YEAR_TO + 1):
        h, docs, _ = search_all({**params,
                                 "EntscheidungsdatumVon": f"{yr}-01-01",
                                 "EntscheidungsdatumBis": f"{yr}-12-31"})
        total += h
        for f in docs:
            if f["id"] not in seen:
                seen.add(f["id"])
                out.append(f)
    return total, out, False

print("search_all ready")

search_all ready


## 6. Rechtssatz-layer baseline

What the importer's own query returns when it *is* properly paged — this separates
"lost to the 20-result cap" from "structurally unreachable".

In [7]:
# ---- RS-layer baseline: what the importer's own search *should* have returned ----
YEAR_PARAMS = {"EntscheidungsdatumVon": f"{YEAR_FROM}-01-01",
               "EntscheidungsdatumBis": f"{YEAR_TO}-12-31"}

rs_baseline = {}
print("Rechtssatz-mode (API default) hits per keyword, 2000-2025, properly paged:")
for kw in KEYWORDS_RIS:
    hits, docs, trunc = search_all({"Applikation": RIS_APP, "Suchworte": kw, **YEAR_PARAMS},
                                   label=f"{kw:26s}")
    rs_baseline[kw] = {"hits": hits, "retrieved": len(docs), "truncated": trunc,
                       "ids": [d["id"] for d in docs]}
print(f"\nrequests so far: {REQUESTS_MADE}")

Rechtssatz-mode (API default) hits per keyword, 2000-2025, properly paged:
    Entfremdung               : Hits=   19  retrieved=   19
    elterliche Entfremdung    : Hits=    0  retrieved=    0
    Eltern-Kind-Entfremdung   : Hits=    0  retrieved=    0
    Kindeswohlgefährdung      : Hits=   16  retrieved=   16
    Loyalitätskonflikt        : Hits=    1  retrieved=    1
    Kontaktverweigerung       : Hits=    0  retrieved=    0

requests so far: 17


## 7. Main probe (A) — full-text decision search

In [8]:
# ---- MAIN PROBE A: full-text (Entscheidungstext) search per keyword ---------
ft = {}          # keyword -> {"hits":, "docs":[fields]}
print("Full-text decision search (Dokumenttyp.SucheInEntscheidungstexten=true), 2000-2025:")
for kw in KEYWORDS_RIS:
    params = {"Applikation": RIS_APP, "Suchworte": kw, **TE_PARAM, **YEAR_PARAMS}
    hits, docs, trunc = search_all(params, label=f"{kw:26s}")
    if trunc:
        print(f"      paging capped out — falling back to year-windowing for '{kw}'")
        hits, docs, trunc = year_window(params)
        print(f"      year-windowed: {len(docs)} distinct decisions")
    # keep only Entscheidungstexte inside the year range (defensive)
    docs = [d for d in docs
            if d["dokumenttyp"] == "Text"
            and (year_of(d) is None or YEAR_FROM <= year_of(d) <= YEAR_TO)]
    ft[kw] = {"hits": hits, "truncated": trunc, "docs": docs}

all_ft = {}
for kw, v in ft.items():
    for d in v["docs"]:
        all_ft.setdefault(d["id"], {**d, "keywords": set()})["keywords"].add(kw)
print(f"\ndistinct full-text decisions across all keywords: {len(all_ft)}")
print(f"requests so far: {REQUESTS_MADE}")

Full-text decision search (Dokumenttyp.SucheInEntscheidungstexten=true), 2000-2025:
    Entfremdung               : Hits=  510  retrieved=  510
    elterliche Entfremdung    : Hits=   11  retrieved=   11
    Eltern-Kind-Entfremdung   : Hits=    1  retrieved=    1
    Kindeswohlgefährdung      : Hits=  243  retrieved=  243
    Loyalitätskonflikt        : Hits=   90  retrieved=   90
    Kontaktverweigerung       : Hits=    5  retrieved=    5

distinct full-text decisions across all keywords: 803
requests so far: 30


## 8. (B) The current corpus

In [9]:
# ---- B: the current corpus (read-only) -------------------------------------
corpus = json.load(open(CORPUS, encoding="utf-8"))
corpus_te = [r for r in corpus if r.get("dokumenttyp") == "Text"]
corpus_rs = [r for r in corpus if r.get("dokumenttyp") == "Rechtssatz"]
corpus_ids = {r["id"] for r in corpus_te if r.get("id")}
corpus_gz  = {norm_gz(r.get("geschaeftszahl")): r["id"] for r in corpus_te if r.get("geschaeftszahl")}

print(f"corpus: {len(corpus)} records = {len(corpus_rs)} Rechtssätze + {len(corpus_te)} decisions")
print(f"decisions by matched keyword:")
kwc = collections.Counter(k for r in corpus_te for k in (r.get("matched_keywords") or []))
for kw in KEYWORDS_RIS:
    print(f"    {kw:26s} {kwc.get(kw, 0):4d}")
print(f"corpus decisions flagged criminal by the audit rule: "
      f"{sum(1 for r in corpus_te if is_criminal(r))}")

def in_corpus(rec):
    """Identity: Dokumentnummer first, normalised Geschäftszahl as fallback."""
    if rec["id"] in corpus_ids:
        return True
    g = norm_gz(rec.get("geschaeftszahl"))
    return bool(g) and g in corpus_gz

corpus: 548 records = 38 Rechtssätze + 510 decisions
decisions by matched keyword:
    Entfremdung                 139
    elterliche Entfremdung        0
    Eltern-Kind-Entfremdung       0
    Kindeswohlgefährdung        377
    Loyalitätskonflikt            3
    Kontaktverweigerung           0
corpus decisions flagged criminal by the audit rule: 25


## 9. (C) + (D) — the recall gap, split civil / criminal

Identity is the RIS Dokumentnummer, with a normalised Geschäftszahl as fallback.
Civil/criminal uses the same rule as `corpus_audit.ipynb`: `Rechtsgebiete` contains
*Strafrecht*, or an `Os` senate marker in the Geschäftszahl.

In [10]:
# ---- C + D: recall gap, split civil / criminal ------------------------------
rows, missed_all = [], {}
for kw in KEYWORDS_RIS:
    docs = ft[kw]["docs"]
    civil = [d for d in docs if not is_criminal(d)]
    crim  = [d for d in docs if is_criminal(d)]
    overlap = [d for d in docs if in_corpus(d)]
    missed  = [d for d in docs if not in_corpus(d)]
    missed_civil = [d for d in missed if not is_criminal(d)]
    for d in missed_civil:
        missed_all.setdefault(d["id"], {**d, "keywords": set()})["keywords"].add(kw)
    rows.append({
        "keyword": kw,
        "corpus_decisions": kwc.get(kw, 0),
        "fulltext_total": len(docs),
        "fulltext_civil": len(civil),
        "overlap": len(overlap),
        "missed_civil": len(missed_civil),
        "criminal_hits": len(crim),
        "missed_total": len(missed),
        "rs_hits_in_range": rs_baseline[kw]["hits"],
    })

# distinct totals (a decision matching two keywords is counted once)
d_all      = list(all_ft.values())
d_civil    = [d for d in d_all if not is_criminal(d)]
d_crim     = [d for d in d_all if is_criminal(d)]
d_overlap  = [d for d in d_all if in_corpus(d)]
d_missed_c = [d for d in d_civil if not in_corpus(d)]

TOTAL = {"keyword": "TOTAL (distinct)",
         "corpus_decisions": len(corpus_te),
         "fulltext_total": len(d_all),
         "fulltext_civil": len(d_civil),
         "overlap": len(d_overlap),
         "missed_civil": len(d_missed_c),
         "criminal_hits": len(d_crim),
         "missed_total": len([d for d in d_all if not in_corpus(d)]),
         "rs_hits_in_range": sum(r["rs_hits_in_range"] for r in rows)}

hdr = f"{'keyword':26s} {'corpus':>7s} {'FT tot':>7s} {'FT civ':>7s} {'overlap':>8s} {'MISSED civ':>11s} {'crim':>6s}"
print(hdr); print("-" * len(hdr))
for r in rows + [TOTAL]:
    if r is TOTAL:
        print("-" * len(hdr))
    print(f"{r['keyword']:26s} {r['corpus_decisions']:7d} {r['fulltext_total']:7d} "
          f"{r['fulltext_civil']:7d} {r['overlap']:8d} {r['missed_civil']:11d} {r['criminal_hits']:6d}")

# sanity: corpus decisions the full-text sweep did not re-find
found_ids = set(all_ft)
found_gz  = {norm_gz(d.get("geschaeftszahl")) for d in d_all}
not_refound = [r for r in corpus_te
               if r["id"] not in found_ids and norm_gz(r.get("geschaeftszahl")) not in found_gz]
print(f"\nsanity — corpus decisions NOT re-found by the full-text sweep: {len(not_refound)}"
      f" ({sum(1 for r in not_refound if is_criminal(r))} of them criminal)")

keyword                     corpus  FT tot  FT civ  overlap  MISSED civ   crim
------------------------------------------------------------------------------
Entfremdung                    139     510     121       50          81    389
elterliche Entfremdung           0      11      11        4           7      0
Eltern-Kind-Entfremdung          0       1       1        0           1      0
Kindeswohlgefährdung           377     243     239      114         125      4
Loyalitätskonflikt               3      90      86       33          53      4
Kontaktverweigerung              0       5       5        0           5      0
------------------------------------------------------------------------------
TOTAL (distinct)               510     803     408      174         244    395

sanity — corpus decisions NOT re-found by the full-text sweep: 338 (15 of them criminal)


### 9b. The other side of the ledger — precision

In [11]:
# ---- the other side of the ledger: precision of the RS-anchored expansion ---
# 'not re-found' corpus decisions are decisions pulled in via a Rechtssatz link
# whose own text may never contain the search phrase. Checked locally, no requests.
def contains_kw(rec, kw):
    return kw.casefold() in (rec.get("full_text") or "").casefold()

no_kw = [r for r in corpus_te
         if not any(contains_kw(r, k) for k in (r.get("matched_keywords") or []))]
print(f"corpus decisions whose own full text never contains any of their matched keywords: "
      f"{len(no_kw)} / {len(corpus_te)}")
print(f"  ... of the {len(not_refound)} not re-found by the full-text sweep: "
      f"{sum(1 for r in not_refound if r in no_kw)}")
print("=> the Rechtssatz-anchored expansion is imprecise as well as incomplete: it imports")
print("   every decision linked to a matching headnote, phrase present in the decision or not.")

corpus decisions whose own full text never contains any of their matched keywords: 368 / 510
  ... of the 338 not re-found by the full-text sweep: 338
=> the Rechtssatz-anchored expansion is imprecise as well as incomplete: it imports
   every decision linked to a matching headnote, phrase present in the decision or not.


## 10. Spot-check — why was each missed decision missed?

In [12]:
# ---- spot-check: WHY was each missed decision missed? -----------------------
# Three mutually exclusive causes, decided per sampled decision:
#   (a) no-Rechtssatz      — the decision has no Rechtssatz at all, so an RS-only
#                            search can never reach it;
#   (b) headnote mismatch  — it has Rechtssätze, but none of them turns up in the
#                            properly-paged RS search for that keyword (the phrase
#                            is in the reasoning, not in the headnote);
#   (c) RS-cap truncation  — one of its Rechtssätze *is* in the paged RS result set,
#                            so only the importer's 20-per-window cap lost it.
import random

rs_ids_by_kw = {kw: set(v["ids"]) for kw, v in rs_baseline.items()}

def rs_for_decision(gz_norm):
    """Rechtssätze that list this Geschäftszahl (RS-mode search on the case number)."""
    if not gz_norm:
        return []
    _, docs, _ = search_all({"Applikation": RIS_APP, "Suchworte": gz_norm})
    hit = []
    for d in docs:
        parts = {norm_gz(p) for p in str(d.get("geschaeftszahl", "")).split(";")}
        if gz_norm in parts:
            hit.append(d)
    return hit

missed_list = sorted(missed_all.values(), key=lambda d: d["id"])
random.seed(20260826)
sample = random.sample(missed_list, min(12, len(missed_list)))

spot = []
print(f"spot-checking {len(sample)} of {len(missed_list)} missed civil decisions\n")
for d in sample:
    g = norm_gz(d["geschaeftszahl"])
    rss = rs_for_decision(g)
    if not rss:
        cause = "a: no-Rechtssatz"
    else:
        kws = d["keywords"]
        if any(r["id"] in set().union(*[rs_ids_by_kw[k] for k in kws]) for r in rss):
            cause = "c: RS-cap truncation"
        else:
            cause = "b: headnote mismatch"
    spot.append({"id": d["id"], "geschaeftszahl": d["geschaeftszahl"],
                 "entscheidungsdatum": d["entscheidungsdatum"],
                 "keywords": sorted(d["keywords"]), "n_rechtssaetze": len(rss),
                 "cause": cause})
    print(f"  {g:16s} {d['entscheidungsdatum'][:10]}  RS={len(rss):3d}  "
          f"{','.join(sorted(d['keywords']))[:40]:40s} -> {cause}")

cause_counts = collections.Counter(s["cause"] for s in spot)
print("\ncauses in the sample:", dict(cause_counts))
print(f"requests so far: {REQUESTS_MADE}")

spot-checking 12 of 244 missed civil decisions

  4ob95/18a        2018-06-11  RS=  1  Kindeswohlgefährdung                     -> b: headnote mismatch
  9ob12/19h        2019-03-28  RS= 14  Loyalitätskonflikt                       -> b: headnote mismatch
  8ob30/20z        2020-07-29  RS=  2  Kindeswohlgefährdung,Loyalitätskonflikt  -> b: headnote mismatch
  4ob146/03d       2003-07-08  RS=  3  Kindeswohlgefährdung                     -> b: headnote mismatch
  9ob41/24f        2024-04-24  RS=  0  Kindeswohlgefährdung                     -> a: no-Rechtssatz
  5ob165/04g       2004-09-14  RS=  1  Loyalitätskonflikt                       -> b: headnote mismatch
  6ob177/20b       2020-10-22  RS=  4  Kindeswohlgefährdung                     -> b: headnote mismatch
  bsw13006/13      2014-09-18  RS=  3  Loyalitätskonflikt                       -> b: headnote mismatch
  7ob252/09y       2010-03-17  RS=  2  Loyalitätskonflikt                       -> b: headnote mismatch
  9ob3/19k         2

## 11. Findings

In [13]:
# ---- findings, composed from the measured numbers --------------------------
gap_ratio  = TOTAL["missed_civil"] / max(len(corpus_te), 1)
civil_share = TOTAL["fulltext_civil"] / max(TOTAL["fulltext_total"], 1)
dom = cause_counts.most_common(1)[0] if cause_counts else ("n/a", 0)
n_a = cause_counts.get("a: no-Rechtssatz", 0)
n_b = cause_counts.get("b: headnote mismatch", 0)
n_c = cause_counts.get("c: RS-cap truncation", 0)

FINDINGS = (
 f"A full-text search of the Entscheidungstexte finds {TOTAL['fulltext_total']} distinct "
 f"decisions for the same six keywords over {YEAR_FROM}–{YEAR_TO}, of which "
 f"{TOTAL['fulltext_civil']} are civil; the corpus currently holds {len(corpus_te)} decisions "
 f"and overlaps with only {TOTAL['overlap']} of them, leaving **{TOTAL['missed_civil']} civil "
 f"decisions missed** — a recall gap of roughly {gap_ratio:.0%} of the present decision count. "
 f"In the spot-check the gap is dominated by *{dom[0].split(': ')[1]}* "
 f"({n_a} no-Rechtssatz, {n_b} headnote-wording mismatch, {n_c} lost to the 20-per-window cap "
 f"of {len(spot)} sampled), i.e. the phrase sits in the reasoning or the decision was never "
 f"headnoted at all, which an RS-only search cannot reach by construction. "
 f"Paging does work with the documented parameter names (`DokumenteProSeite=OneHundred` + "
 f"`Seitennummer`, {t0} Hits fully retrievable where the unpaged call returns {len(d0)}), so the "
 f"20-results ceiling was a parameter-name bug, not an API limit, so the year-windowing "
 f"workaround is unnecessary — though because the in-range Rechtssatz hit counts are small "
 f"({', '.join(str(rs_baseline[k]['hits']) for k in KEYWORDS_RIS)}), the windowing did recover "
 f"nearly all of it and no sampled miss traces to the 20-per-window cap. "
 f"The homonym cost of naively enabling the full-text flag is large: "
 f"{TOTAL['criminal_hits']} of the {TOTAL['fulltext_total']} full-text hits "
 f"({1 - civil_share:.0%}) are criminal-senate *Entfremdung* (misappropriation), so the flag "
 f"cannot be switched on without keeping the civil/criminal filter in front of it. "
 f"The same comparison exposes a precision cost in the opposite direction: {len(no_kw)} of the "
 f"{len(corpus_te)} decisions already in the corpus never contain any of their own matched "
 f"keywords, because the Rechtssatz-anchored expansion imports every decision linked to a "
 f"matching headnote."
)
print(FINDINGS)

A full-text search of the Entscheidungstexte finds 803 distinct decisions for the same six keywords over 2000–2025, of which 408 are civil; the corpus currently holds 510 decisions and overlaps with only 174 of them, leaving **244 civil decisions missed** — a recall gap of roughly 48% of the present decision count. In the spot-check the gap is dominated by *headnote mismatch* (2 no-Rechtssatz, 10 headnote-wording mismatch, 0 lost to the 20-per-window cap of 12 sampled), i.e. the phrase sits in the reasoning or the decision was never headnoted at all, which an RS-only search cannot reach by construction. Paging does work with the documented parameter names (`DokumenteProSeite=OneHundred` + `Seitennummer`, 36 Hits fully retrievable where the unpaged call returns 20), so the 20-results ceiling was a parameter-name bug, not an API limit, so the year-windowing workaround is unnecessary — though because the in-range Rechtssatz hit counts are small (19, 0, 0, 16, 1, 0), the windowing did reco

## 12. Write the reports

In [14]:
# ---- write reports/ris_recall_probe.json + .md ------------------------------
REPORT_DIR.mkdir(exist_ok=True)
stamp = datetime.now().isoformat(timespec="seconds")

payload = {
    "generated": stamp,
    "attribution": "RIS, Bundeskanzleramt Österreich (CC BY 4.0) — https://data.bka.gv.at",
    "endpoint": RIS_API,
    "applikation": RIS_APP,
    "years": [YEAR_FROM, YEAR_TO],
    "keywords": KEYWORDS_RIS,
    "requests_made": REQUESTS_MADE,
    "paging": {
        "works": bool(PAGING_WORKS),
        "params": {"DokumenteProSeite": PAGE_SIZE, "Seitennummer": "1..n"},
        "ignored_param_in_production_note": "Seite",
        "pages_probe": [{"page": p, "hits": h, "returned": n, "new": nw, "cumulative": c}
                        for p, h, n, nw, c in pages],
        "no_param_returned": len(d0), "no_param_hits": t0,
        "seite_param_returned": len(d_s),
        "paged_single_call_returned": len(d_p),
    },
    "dokumenttyp_encoding": {
        "working": "Dokumenttyp.SucheInEntscheidungstexten=true",
        "tested": encoding_results,
    },
    "rs_baseline": {k: {kk: vv for kk, vv in v.items() if kk != "ids"}
                    for k, v in rs_baseline.items()},
    "per_keyword": rows,
    "total": TOTAL,
    "corpus": {"file": str(CORPUS), "records": len(corpus),
               "rechtssaetze": len(corpus_rs), "decisions": len(corpus_te),
               "decisions_criminal_by_rule": sum(1 for r in corpus_te if is_criminal(r)),
               "decisions_without_own_keyword_in_text": len(no_kw)},
    "missed_civil_decisions": [
        {"id": d["id"], "geschaeftszahl": d["geschaeftszahl"],
         "entscheidungsdatum": d["entscheidungsdatum"], "gericht": d["gericht"],
         "rechtsgebiete": d["rechtsgebiete"], "keywords": sorted(d["keywords"]),
         "dokument_url": d["dokument_url"]}
        for d in missed_list],
    "missed_criminal_count": len([d for d in d_crim if not in_corpus(d)]),
    "corpus_not_refound": [{"id": r["id"], "geschaeftszahl": r.get("geschaeftszahl"),
                            "criminal": is_criminal(r)} for r in not_refound],
    "spot_check": spot,
    "spot_check_causes": dict(cause_counts),
}
(REPORT_DIR / "ris_recall_probe.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"wrote {REPORT_DIR / 'ris_recall_probe.json'}")


def md_table(rows, total):
    head = ("| keyword | corpus decisions (RS-anchored) | full-text decisions (total) | "
            "full-text civil | overlap | MISSED civil | criminal-homonym hits |\n"
            "|---|---:|---:|---:|---:|---:|---:|\n")
    body = "".join(
        f"| {r['keyword']} | {r['corpus_decisions']} | {r['fulltext_total']} | "
        f"{r['fulltext_civil']} | {r['overlap']} | {r['missed_civil']} | {r['criminal_hits']} |\n"
        for r in rows)
    body += (f"| **{total['keyword']}** | **{total['corpus_decisions']}** | "
             f"**{total['fulltext_total']}** | **{total['fulltext_civil']}** | "
             f"**{total['overlap']}** | **{total['missed_civil']}** | "
             f"**{total['criminal_hits']}** |\n")
    return head + body

MD = f"""# RIS recall probe — how much does the Rechtssatz-only ingest miss?

*Generated {stamp} · read-only diagnostic (`src/ris_recall_probe.ipynb`); the production
corpus `data/ris_parental_alienation.json` and `src/master_import.ipynb` were not modified.*

Data: **RIS, Bundeskanzleramt Österreich (CC BY 4.0)** — <https://data.bka.gv.at>.
{REQUESTS_MADE} API requests, {REQUEST_DELAY}s apart, 6 keywords, {YEAR_FROM}–{YEAR_TO}.

## What the production importer sends

`import_ris` calls `ris_get` with `{{"Applikation": "Justiz", "Suchworte": kw}}` plus, for
keywords with >20 hits, `EntscheidungsdatumVon`/`Bis` year windows. It sends **no
`Dokumenttyp`** — so the API default applies and only **Rechtssätze** are searched — and
**no paging parameters at all**. Decisions enter the corpus only by following each matched
Rechtssatz's linked `Entscheidungstexte`.

## Confirmation 1 — paging works; the old note was a parameter-name bug

| query | Hits reported | documents returned |
|---|---:|---:|
| no paging params | {t0} | {len(d0)} |
| `Seite=2` (the name the old note assumed) | {t_s} | {len(d_s)} |
| `DokumenteProSeite=OneHundred&Seitennummer=1` | {t_p} | {len(d_p)} |

Same keyword, full-text variant (a genuinely multi-page result set):

| page | Hits reported | returned | new (not seen on earlier pages) |
|---:|---:|---:|---:|
""" + "".join(f"| {p} | {h} | {n} | {nw} |\n" for p, h, n, nw, c in pages) + f"""

The documented v2.6 names — **`DokumenteProSeite` ("Ten"/"Twenty"/"Fifty"/"OneHundred")** and
**`Seitennummer`** — page correctly and return disjoint document sets. `Seite` is silently
ignored, which is what produced the "always 20, ignores the page parameter" note. A page past
the end of the result set answers HTTP 500; that is the end-of-results signal, not a failure.
**The year-windowing workaround is therefore unnecessary.** Whether its residual 20-per-window
truncation actually cost anything is a separate question, answered by the RS-layer baseline
below: the in-range Rechtssatz hit counts are small (see `rs_baseline`), so windowing recovered
nearly all of them, and no decision in the spot-check traces its absence to the cap.

## Confirmation 2 — the Dokumenttyp flag is a nested parameter

| encoding | Hits | doctypes on page 1 |
|---|---:|---|
""" + "".join(f"| `{e['encoding']}` | {e['hits']} | {e['doctypes']} |\n" for e in encoding_results) + f"""

Only the **nested** form `Dokumenttyp.SucheInEntscheidungstexten=true` switches the search;
it returns `Dokumenttyp: "Text"` (Entscheidungstexte). The flat spellings are ignored and fall
back to the Rechtssatz default.

## The recall gap

Full-text decision search (`Dokumenttyp.SucheInEntscheidungstexten=true`, {YEAR_FROM}–{YEAR_TO},
fully paged) against the decisions currently in the corpus (`dokumenttyp == "Text"`, matched on
Dokumentnummer with normalised Geschäftszahl as fallback). Civil/criminal uses the corpus-audit
rule: `Rechtsgebiete` contains *Strafrecht*, or an `Os` senate marker in the Geschäftszahl.

{md_table(rows, TOTAL)}
Rows count per keyword, so a decision matching two keywords is counted twice; the TOTAL row is
distinct decisions. Corpus decisions not re-found by the full-text sweep: **{len(not_refound)}**
({sum(1 for r in not_refound if is_criminal(r))} criminal).

The other side of the ledger: **{len(no_kw)}** of the {len(corpus_te)} corpus decisions never
contain any of their own matched keywords in their text at all. The Rechtssatz-anchored
expansion imports every decision linked to a matching headnote whether or not the phrase
occurs in that decision — so the present ingest is imprecise as well as incomplete, and that
is why {len(not_refound)} corpus decisions are not re-found by a search on the phrase itself.

### Why each missed decision was missed (sample of {len(sample)})

| Geschäftszahl | date | Rechtssätze | keyword(s) | cause |
|---|---|---:|---|---|
""" + "".join(
    f"| {norm_gz(s['geschaeftszahl'])} | {s['entscheidungsdatum'][:10]} | {s['n_rechtssaetze']} | "
    f"{', '.join(s['keywords'])} | {s['cause']} |\n" for s in spot) + f"""

Causes: **a** = the decision has no Rechtssatz at all (unreachable by an RS-only search);
**b** = it has Rechtssätze but none of them surfaces in the properly-paged RS search for that
keyword (the phrase lives in the reasoning, not the headnote); **c** = one of its Rechtssätze
*is* in the paged RS result set, so only the importer's 20-per-window cap lost it.
Sample: {dict(cause_counts)}.

## Findings

{FINDINGS}
"""
(REPORT_DIR / "ris_recall_probe.md").write_text(MD, encoding="utf-8")
print(f"wrote {REPORT_DIR / 'ris_recall_probe.md'}")

wrote ../reports/ris_recall_probe.json
wrote ../reports/ris_recall_probe.md


## 14. Appendix — keyword scoping

The probe above closed the recall gap *for the six configured keywords*, worth ~170 usable
decisions. That is small for 25 years of Austrian family law, which raises the prior question:
are those the right six? `KEYWORDS["ris"]` was identical to `KEYWORDS["swiss"]`, and the
config's reason for dropping broad terms ("migration/admin/social courts flood the corpus")
cannot apply to RIS — `Applikation=Justiz` is ordinary jurisdiction by construction.

Written up in `reports/ris_keyword_scope.md`.

In [15]:
# ---- A1: candidate terms, full-text hit counts -----------------------------
# The recall probe closed the gap for the six configured keywords. This asks the
# prior question: are those the right six? Hits only — one request per term.
CANDIDATES = ["Obsorge", "Kindeswohl", "Pflege und Erziehung", "Kontaktrecht",
              "Besuchsrecht", "Obhut", "Obsorgeübertragung", "Umgangsrecht"]

def ft_hits(kw, **extra):
    t, _ = ris_parse(ris_get({"Applikation": RIS_APP, "Suchworte": kw, **TE_PARAM,
                              **YEAR_PARAMS, "DokumenteProSeite": "Ten",
                              "Seitennummer": 1, **extra}))
    return t

print("full-text decision hits, Justiz, 2000-2025\n")
print("--- the six keywords currently in KEYWORDS['ris'] ---")
for kw in KEYWORDS_RIS:
    print(f"  {kw:26s} {ft_hits(kw)}")
print("\n--- broad family terms dropped from the config ---")
for kw in CANDIDATES:
    print(f"  {kw:26s} {ft_hits(kw)}")

full-text decision hits, Justiz, 2000-2025

--- the six keywords currently in KEYWORDS['ris'] ---
  Entfremdung                510
  elterliche Entfremdung     11
  Eltern-Kind-Entfremdung    1
  Kindeswohlgefährdung       243
  Loyalitätskonflikt         90
  Kontaktverweigerung        5

--- broad family terms dropped from the config ---
  Obsorge                    2409
  Kindeswohl                 1220
  Pflege und Erziehung       813
  Kontaktrecht               505
  Besuchsrecht               436
  Obhut                      292
  Obsorgeübertragung         199
  Umgangsrecht               32


In [16]:
# ---- A2: exact union of all candidate terms, and its precision -------------
def ft_sweep(kw):
    _, docs, _ = search_all({"Applikation": RIS_APP, "Suchworte": kw, **TE_PARAM, **YEAR_PARAMS})
    return {d["id"]: d for d in docs if d["dokumenttyp"] == "Text"}

UNION = {}
for kw in KEYWORDS_RIS + CANDIDATES:
    for k, v in ft_sweep(kw).items():
        UNION.setdefault(k, {**v, "kws": set()})["kws"].add(kw)

keep = lambda v: "strafrecht" not in (v["rechtsgebiete"] or "").lower() \
                 and not (v["gericht"] or "").startswith("AUSL")
clean = {k: v for k, v in UNION.items() if keep(v)}
print(f"union (all terms)                  {len(UNION)}")
print(f"  criminal (Rechtsgebiete)            "
      f"{sum(1 for v in UNION.values() if 'strafrecht' in (v['rechtsgebiete'] or '').lower())}")
print(f"  AUSL EGMR summaries                 "
      f"{sum(1 for v in UNION.values() if (v['gericht'] or '').startswith('AUSL'))}")
print(f"union, civil & domestic            {len(clean)}")
print(f"  NEW vs corpus                       {sum(1 for k in clean if k not in corpus_ids)}")

print("\nmarginal NEW civil-domestic decisions each broad term adds, greedily:")
base = {k for k, v in clean.items() if any(w in v["kws"] for w in KEYWORDS_RIS)}
print(f"  [current 6 keywords]      base = {len(base)}")
pool, acc = set(CANDIDATES), set(base)
while pool:
    best = max(pool, key=lambda w: len({k for k, v in clean.items() if w in v["kws"]} - acc))
    add = {k for k, v in clean.items() if best in v["kws"]} - acc
    acc |= add; pool.discard(best)
    print(f"  + {best:22s} +{len(add):5d}  -> {len(acc):5d}")

union (all terms)                  3874
  criminal (Rechtsgebiete)            546
  AUSL EGMR summaries                 177
union, civil & domestic            3151
  NEW vs corpus                       2705

marginal NEW civil-domestic decisions each broad term adds, greedily:
  [current 6 keywords]      base = 389
  + Obsorge                + 2009  ->  2398
  + Pflege und Erziehung   +  326  ->  2724
  + Kindeswohl             +  206  ->  2930
  + Kontaktrecht           +   77  ->  3007
  + Obhut                  +   74  ->  3081
  + Besuchsrecht           +   69  ->  3150
  + Obsorgeübertragung     +    1  ->  3151
  + Umgangsrecht           +    0  ->  3151


The union of 3,151 is a mirage: spot-checked by KWIC, decisions matched only by bare
"Obsorge" were on-topic 2/6 — the rest use it in the case caption. Bare "Entfremdung" was
0/6. Caption-anchored hits were 6/6.

In [17]:
# ---- A3: Suchworte is a PHRASE match -> the caption convention is a filter --
# Austrian decisions state their subject in the caption: "Pflegschaftssache ...
# wegen Obsorge". Bare "Obsorge" also matches the party description of any
# proceeding involving a minor ("beide in Obsorge der Mutter"), which is boilerplate.
for kw in ["Obsorge", "wegen Obsorge", "Kontaktrecht", "wegen Kontaktrecht",
           "wegen Unterhalt", "Obsorge und Kontaktrecht"]:
    print(f"  {kw:30s} {ft_hits(kw)}")

CAPTION = ["wegen Obsorge", "wegen Kontaktrecht", "wegen Besuchsrecht"]
CAP = {}
for kw in CAPTION:
    for k, v in ft_sweep(kw).items():
        CAP.setdefault(k, {**v, "kws": set()})["kws"].add(kw)

# the 2013 KindNamRÄG renamed Besuchsrecht -> Kontaktrecht
print("\nterminology break at the KindNamRÄG 2013:")
for term in ["wegen Besuchsrecht", "wegen Kontaktrecht"]:
    yrs = [year_of(v) for v in CAP.values() if term in v["kws"]]
    print(f"  {term:22s} 2000-2012: {sum(1 for y in yrs if y <= 2012):4d}"
          f"   2013-2025: {sum(1 for y in yrs if y >= 2013):4d}")

TOPIC = [k for k in KEYWORDS_RIS if k != "Entfremdung"]
adopted = {k for k, v in CAP.items() if keep(v)} | \
          {k for k, v in clean.items() if any(w in v["kws"] for w in TOPIC)}
print(f"\nadopted scope (caption-anchored + topic terms, civil & domestic): {len(adopted)}")
print(f"  already in corpus {len(adopted & corpus_ids)}   NEW {len(adopted - corpus_ids)}")

  Obsorge                        2409
  wegen Obsorge                  1796
  Kontaktrecht                   505
  wegen Kontaktrecht             425
  wegen Unterhalt                3514
  Obsorge und Kontaktrecht       373

terminology break at the KindNamRÄG 2013:
  wegen Besuchsrecht     2000-2012:  201   2013-2025:   88
  wegen Kontaktrecht     2000-2012:    1   2013-2025:  424

adopted scope (caption-anchored + topic terms, civil & domestic): 1955
  already in corpus 339   NEW 1616


**Adopted:** caption-anchored terms + the topic-specific ones, filtered to civil and
domestic — **1,955 decisions, 1,616 new**. Bare `Entfremdung` stays on the Rechtssatz route
only. Implemented in `master_import.ipynb` (`RIS_FULLTEXT_KEYWORDS`); every record now carries
`match_route` so the two ingest routes stay separable downstream.

## 13. Plain findings

Enabling a full-text decision search finds **803 distinct decisions** for the same six keywords
over 2000–2025 where the corpus holds 510, and only **174** of those 803 are already in it —
**244 civil decisions are missed**, roughly half again the civil decision count the pipeline
currently has. The gap is overwhelmingly a **headnote-wording mismatch**, not missing headnotes:
in the 12-decision spot-check 10 had Rechtssätze that simply never use the search phrase
(*Loyalitätskonflikt* and *Kindeswohlgefährdung* live in the reasoning, while the headnote is
phrased in Obsorge/Kontaktrecht terms) and only 2 had no Rechtssatz at all — so this is a
structural limit of searching headnotes, not a coverage accident. **Paging does work**:
`DokumenteProSeite=OneHundred` + `Seitennummer` returns all 36 Hits in one call and 100
distinct documents per page on an 807-hit query, whereas the assumed `Seite` parameter is
silently ignored — the "20 per page" ceiling was a parameter-name bug, so the year-windowing
workaround is unnecessary, though because the Rechtssatz hit counts are small (19 / 16 / 1 in
range) the windowing did in practice recover almost all of it and no sampled miss traces to
the cap. The homonym cost of naively flipping the flag on is severe: **395 of the 803 full-text
hits (49 %) are criminal-senate *Entfremdung*** (misappropriation of property), so the
full-text search cannot be adopted without the civil/criminal filter in front of it.
Finally, the diff cuts both ways — **368 of the 510 decisions already in the corpus never
contain any of their own matched keywords**, because the Rechtssatz-anchored expansion imports
every decision linked to a matching headnote regardless of its own wording.

*No production data was changed by this notebook.*